In [ ]:
import yfinance as yf
df = yf.download("AAPL", start="2015-01-01", end="2024-11-01")


/tmp/ipykernel_7197/2096725268.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("AAPL", start="2015-01-01", end="2024-11-01")
[*********************100%***********************]  1 of 1 completed


In [8]:
data = df[['Close']]


In [ ]:
data.shape
data

(2475, 1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Assuming "data" contains only the Close column
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data.values)

print("Scaled shape:", scaled_data.shape)


Scaled shape: (2475, 1)


In [14]:
scaled_data

array([[0.01693515],
       [0.01375268],
       [0.01376299],
       ...,
       [0.98697638],
       [0.97043022],
       [0.95101057]], shape=(2475, 1))

In [15]:
sequence_length = 60

X = []
y = []

for i in range(sequence_length, len(scaled_data)):
    X.append(scaled_data[i-sequence_length:i, 0])   # previous 60 days
    y.append(scaled_data[i, 0])                     # next day

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

# Reshape X → (samples, 60, 1) for RNN/LSTM input
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

print("X shape:", X.shape)   # (samples, 60, 1)
print("y shape:", y.shape)   # (samples,)


X shape: (2415, 60, 1)
y shape: (2415,)


Vanilla RNN Model

In [17]:
!pip3 install tensorflow

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.4/620.4 MB 167.1 kB/s  0:47:290:00:02

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

# Build the Vanilla RNN model
rnn_model = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=(60, 1)),
    Dense(1)
])

# Compile the model
rnn_model.compile(optimizer='adam', loss='mse')

# Model summary
rnn_model.summary()
